In [1]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from scipy.spatial.distance import cdist
# from xgboost import XGBRegressor

# # ۱. خواندن فایل اکسل اصلی
# file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_generator_bearings_deviation_monitoring\deviation_monitoring\dsas_g11_generator_bearings_deviation_monitoring_output1.xlsx'
# try:
#     df_raw = pd.read_excel(file_path)
#     print("فایل اصلی با موفقیت خوانده شد.")
# except Exception as e:
#     print(f"خطا در خواندن فایل: {e}")
#     exit()


# # لیست تمام فیچرهای عددی
# all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
# # لیست سنسورهایی که تارگت هستند
# target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']

# # ۲. مرحله پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت با DBSCAN
# scaler = StandardScaler()
# scaled_data = scaler.fit_transform(df_raw[all_features])

# dbscan = DBSCAN(eps=0.5, min_samples=5)
# labels = dbscan.fit_predict(scaled_data)

# cluster_centers = {}
# for cluster_id in set(labels):
#     if cluster_id != -1:
#         cluster_centers[cluster_id] = scaled_data[labels == cluster_id].mean(axis=0)

# def calculate_distance(row_index):
#     label = labels[row_index]
#     point = scaled_data[row_index].reshape(1, -1)
#     if label != -1:
#         center = cluster_centers[label].reshape(1, -1)
#         return cdist(point, center, metric='euclidean')[0][0]
#     else:
#         if not cluster_centers: return 0.0
#         all_centers = np.array(list(cluster_centers.values()))
#         return np.min(cdist(point, all_centers, metric='euclidean'))

# df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

# # حذف ۱۰ درصد داده‌های با بیشترین فاصله
# df_sorted = df_raw.sort_values(by='distance', ascending=False)
# num_to_remove = int(len(df_sorted)*0.10)
# df_cleaned = df_sorted.iloc[num_to_remove:].copy()

# # سورت زمانی و تفکیک Test/Train
# df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
# df_cleaned = df_cleaned.sort_values(by='date')

# split_idx = int(len(df_cleaned) * 0.80)
# train_df = df_cleaned.iloc[:split_idx].copy()
# test_df = df_cleaned.iloc[split_idx:].copy()

# # ۳. شروع حلقه پیش‌بینی برای هر سنسور و اضافه کردن به یک دیتافریم واحد
# print("شروع فرآیند یادگیری و پیش‌بینی برای تمام سنسورها...")

# for target in target_sensors:
#     # تعیین فیچرها (همه بجز تارگت فعلی)
#     current_features = [f for f in all_features if f != target]

#     X_train, y_train = train_df[current_features], train_df[target]
#     X_test = test_df[current_features]

#     # آموزش مدل XGBoost
#     model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)

#     model.fit(X_train, y_train)

#     # ثبت پیش‌بینی در یک ستون جدید با نام اختصاصی
#     column_name = f'predicted_{target.split("_")[1]}'
#     test_df[column_name] = model.predict(X_test)
#     print(f"پیش‌بینی {target} با موفقیت انجام شد.")

# # ۴. ذخیره نهایی در یک فایل اکسل واحد
# # مرتب‌سازی ستون‌ها برای خوانایی بهتر (ابتدا تاریخ، سپس سنسورهای واقعی و در انتها پیش‌بینی‌ها)
# predicted_cols = [f'predicted_{t.split("_")[1]}' for t in target_sensors]
# final_columns_order = ['date'] + all_features + predicted_cols

# test_df[final_columns_order].to_excel(output_filename, index=False)

# print(f"\nعملیات با موفقیت پایان یافت. فایل جامع '{output_filename}' ایجاد شد.")
# print(f"این فایل شامل {len(test_df)} ردیف و ستون‌های پیش‌بینی مجزا برای هر سنسور است.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
from xgboost import XGBRegressor
import os
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

def run_deviation_monitoring():
    """اجرای تحلیل پایش انحراف با XGBoost برای ژنراتور و ذخیره خروجی (فرمت Wide)"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # ۱. خواندن فایل اکسل اصلی
    file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
    output_filename = r'outputs\G11\dsas_g11_generator_bearings_deviation_monitoring\deviation_monitoring\dsas_g11_generator_bearings_deviation_monitoring_output1.xlsx'
    
    # ایجاد پوشه خروجی
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    
    try:
        df_raw = pd.read_excel(file_path)
        print(f"✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return None

    # لیست تمام فیچرهای عددی
    all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                    'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    
    target_sensors = all_features

    # ۲. مرحله پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت با DBSCAN
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])

    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)

    cluster_centers = {}
    for cluster_id in set(labels):
        if cluster_id != -1:
            cluster_centers[cluster_id] = scaled_data[labels == cluster_id].mean(axis=0)

    def calculate_distance(row_index):
        label = labels[row_index]
        point = scaled_data[row_index].reshape(1, -1)
        if label != -1:
            center = cluster_centers[label].reshape(1, -1)
            return cdist(point, center, metric='euclidean')[0][0]
        else:
            if not cluster_centers: return 0.0
            all_centers = np.array(list(cluster_centers.values()))
            return np.min(cdist(point, all_centers, metric='euclidean'))

    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

    # حذف ۱۰ درصد داده‌های با بیشترین فاصله
    df_sorted = df_raw.sort_values(by='distance', ascending=False)
    num_to_remove = int(len(df_sorted)*0.10)
    df_cleaned = df_sorted.iloc[num_to_remove:].copy()
    after_count = len(df_cleaned)
    
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")

    # سورت زمانی و تفکیک Test/Train
    df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
    df_cleaned = df_cleaned.sort_values(by='date')

    split_idx = int(len(df_cleaned) * 0.80)
    train_df = df_cleaned.iloc[:split_idx].copy()
    test_df = df_cleaned.iloc[split_idx:].copy()
    
    print(f"   داده‌های آموزش: {len(train_df):,} رکورد")
    print(f"   داده‌های تست: {len(test_df):,} رکورد")

    # ۳. شروع حلقه پیش‌بینی برای هر سنسور و اضافه کردن به یک دیتافریم واحد
    print("🔄 مرحله 2: یادگیری و پیش‌بینی برای تمام سنسورها...")

    for target in target_sensors:
        # تعیین فیچرها (همه بجز تارگت فعلی)
        current_features = [f for f in all_features if f != target]

        X_train, y_train = train_df[current_features], train_df[target]
        X_test = test_df[current_features]

        # آموزش مدل XGBoost
        model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbosity=0)
        model.fit(X_train, y_train)

        # ثبت پیش‌بینی در یک ستون جدید با نام اختصاصی
        column_name = f'predicted_{target.split("_")[1]}'
        test_df[column_name] = model.predict(X_test)
        
        # محاسبه RMSE
        mse = np.mean((test_df[target] - test_df[column_name]) ** 2)
        rmse = np.sqrt(mse)
        print(f"   ✅ {target}: RMSE = {rmse:.6f}")

    # ۴. ذخیره نهایی در یک فایل اکسل واحد
    print("💾 مرحله 3: ذخیره خروجی...")
    
    # مرتب‌سازی ستون‌ها برای خوانایی بهتر
    predicted_cols = [f'predicted_{t.split("_")[1]}' for t in target_sensors]
    final_columns_order = ['date'] + all_features + predicted_cols

    try:
        test_df[final_columns_order].to_excel(output_filename, index=False)
        print(f"✅ عملیات با موفقیت پایان یافت. فایل جامع '{output_filename}' ایجاد شد.")
        print(f"📊 تعداد رکوردها: {len(test_df):,}")
        print(f"📋 تعداد ستون‌ها: {len(test_df.columns)}")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return test_df

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش انحراف ژنراتور با XGBoost (فرمت Wide)")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["23:11", "23:12", "23:13"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_deviation_monitoring()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه پایش انحراف ژنراتور با XGBoost (فرمت Wide)")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه پایش انحراف ژنراتور با XGBoost (فرمت Wide)
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش انحراف ژنراتور با XGBoost (فرمت Wide)
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-01 23:11:08
🔄 شروع تحلیل در 2026-07-01 23:11:08
✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: 11,915
📅 بازه زمانی: 2021-03-16 05:33:48 تا 2026-05-31 20:30:17
🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...
   حذف 1,191 ردیف به عنوان داده‌های پرت
   داده‌های آموزش: 8,579 رکورد
   داده‌های تست: 2,145 رکورد
🔄 مرحله 2: یادگیری و پیش‌بینی برای تمام سنسورها...
   ✅ AssetID_9362: RMSE = 6.558366
   ✅ AssetID_9363: RMSE = 4.653764
   ✅ AssetID_9364: RMSE = 8.001394
   ✅ AssetID_9365: RMSE = 8.369748
   ✅ AssetID_9366: RMSE = 17.726519
   ✅ AssetID_9367: RMSE = 16.422210
   ✅ AssetID_9371: RMSE = 0.584440
   ✅ AssetID_9372: RMSE = 0.702388
   ✅ AssetID_9373: RMSE = 1.109904
💾 مرحله 3: ذخیره خروجی...
✅ عملیات با موفقی